# Unit 3 solutions: bag-of-words, TF-IDF cosine, and BM25

These reference solutions rebuild the lexical path from the lesson -- a book as a bag of keyword tokens, TF-IDF with cosine similarity, then Okapi BM25 -- reveal the `bookrec` implementation piece by piece, and score it honestly on the Unit 1 scoreboard. A reader's query is always the keyword tokens of the books in their positive `train` history (the `seen` set); `val` is the sealed holdout we grade against. The only random path is the `RandomRetrievalPath(..., seed=0)` floor, so the notebook runs top-to-bottom deterministically. Every exercise ends with an `assert` on a computed result, so a wrong number fails loudly instead of printing quietly.

In [ ]:
import json
from collections import Counter

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])


def tokenize(text):
    return text.split()


# each reader's positive train items == the scoreboard's `seen` set
train_positive = interactions[(interactions["split"] == "train") & (interactions["label"] == 1)]
seen_by = train_positive.groupby("reader_id")["item_id"].apply(set).to_dict()


def profile_of(reader_id):
    # the reader's query: keyword tokens of every book they have read, repeats kept
    return [token for item_id in sorted(seen_by[reader_id]) for token in tokenize(keywords[item_id])]


len(catalog), len(keywords), len(interactions)

## Exercise 1 - Build the bag-of-words view: vocabulary and document frequency

A bag of words forgets word order and keeps only counts. Tokenizing every keyword document gives `tokens_of`; the vocabulary is the sorted set of every token that appears anywhere. Document frequency counts *documents*, not occurrences, so each document contributes its `set(tokens)` to the `Counter` -- a token repeated five times in one book still adds 1 to its `df`.

The bookkeeping identity is a double-counting argument: every (document, token) pair with tf > 0 is counted once when we sum `df` over tokens (column by column) and once when we sum distinct-token counts over documents (row by row), so the two totals must agree. On this corpus no token is confined to a single book -- the keyword generator keeps every token in dozens of books -- so `singleton_tokens` is 0; we print the smallest `df` so the number is not mistaken for a bug.

In [ ]:
tokens_of = {item_id: tokenize(keywords[item_id]) for item_id in catalog_ids}
vocab = sorted({token for tokens in tokens_of.values() for token in tokens})
vocab_size = len(vocab)

df_counter = Counter()
for tokens in tokens_of.values():
    df_counter.update(set(tokens))          # a document counts once per token, however often it repeats it
most_widespread_token, most_widespread_df = df_counter.most_common(1)[0]
singleton_tokens = sum(1 for count in df_counter.values() if count == 1)
smallest_df = min(df_counter.values())

df_total = sum(df_counter.values())
distinct_total = sum(len(set(tokens)) for tokens in tokens_of.values())
identity_holds = df_total == distinct_total

print(f"documents: {len(tokens_of)}   vocabulary: {vocab_size} distinct tokens")
print(f"most widespread token: {most_widespread_token!r} in {most_widespread_df} books "
      f"({most_widespread_df / len(catalog_ids):.1%})")
print(f"tokens in exactly one book: {singleton_tokens}   (smallest df on this corpus: {smallest_df})")
print(f"sum of df = {df_total}   sum of distinct tokens per document = {distinct_total}   "
      f"identity holds: {identity_holds}")

assert identity_holds and df_total == distinct_total
assert vocab_size == len(df_counter) and most_widespread_df == max(df_counter.values())

## Exercise 2 - TF-IDF weights and cosine similarity for a reader's profile, then reveal the library

The document-term matrix `tf` has one row per book (sorted `catalog_ids`) and one column per vocabulary token (alphabetical). Two cross-checks tie it back to Exercise 1: row sums are document lengths, and the count of non-zero entries in each column is that token's `df`. The smoothed IDF

$$\mathrm{idf}_t = \log\!\left(1 + \frac{N - \mathrm{df}_t + 0.5}{\mathrm{df}_t + 0.5}\right)$$

is broadcast across rows, and dividing each weighted row by its L2 norm gives unit-length document vectors. With unit rows, the cosine of a query against every document is the matrix-vector product `tfidf_by_hand @ query` divided by the query norm.

Reader 7's query is the token counts of their four seen books, weighted by the same `idf`. We wrap the cosines in `bookrec.Candidate` objects and let `bookrec.rank(..., exclude=seen)` drop the seen books and take the top 5.

The library stores the same arrays, but numbers its vocabulary in first-seen order rather than alphabetically. `permutation[c]` is the library column that holds our alphabetical column `c`, so `tfidf_lib[:, permutation]` re-orders the library's columns into ours and the two matrices must agree to floating point. The query must be built in the library's column order (and weighted with `index.idf`) before `bookrec.cosine_similarity` can score it.

In [ ]:
n_docs, n_terms = len(catalog_ids), len(vocab)
col_of = {token: col for col, token in enumerate(vocab)}

tf = np.zeros((n_docs, n_terms))
for row, item_id in enumerate(catalog_ids):
    for token in tokens_of[item_id]:
        tf[row, col_of[token]] += 1.0
doc_lengths = np.array([len(tokens_of[item_id]) for item_id in catalog_ids])
df = (tf > 0).sum(axis=0)
tf_checks_pass = bool(np.array_equal(tf.sum(axis=1), doc_lengths)) and all(
    df[col_of[token]] == count for token, count in df_counter.items()
)

idf = np.log(1.0 + (n_docs - df + 0.5) / (df + 0.5))
weighted = tf * idf                                    # idf broadcasts across every row
tfidf_by_hand = weighted / np.linalg.norm(weighted, axis=1, keepdims=True)

reader_id = 7
seen = seen_by[reader_id]
profile_tokens = profile_of(reader_id)
query = np.zeros(n_terms)
for token in profile_tokens:
    query[col_of[token]] += 1.0
query *= idf                                           # the same tf * idf weighting as the documents
cosine_by_hand = (tfidf_by_hand @ query) / np.linalg.norm(query)   # rows already have unit length
top5_tfidf = bookrec.rank(
    [bookrec.Candidate(item_id, float(cosine_by_hand[row]), "tfidf-by-hand")
     for row, item_id in enumerate(catalog_ids)],
    n=5, exclude=seen,
)
top5_tfidf_ids = [c.item_id for c in top5_tfidf]

# reveal the library
index = bookrec.BM25Index(keywords)
permutation = np.array([index.vocab[token] for token in vocab])
tfidf_lib = bookrec.tfidf_matrix(index)
query_lib = np.zeros(len(index.vocab))
for token in profile_tokens:
    query_lib[index.vocab[token]] += 1.0
query_lib *= index.idf
cosine_lib = bookrec.cosine_similarity(query_lib, tfidf_lib)
library_matches = bool(np.allclose(tfidf_lib[:, permutation], tfidf_by_hand)) and bool(
    np.allclose(cosine_lib, cosine_by_hand)
)

print(f"tf matrix {tf.shape}; row-sum and df cross-checks pass: {tf_checks_pass}")
print(f"idf range: {idf.min():.3f} .. {idf.max():.3f}")
print(f"reader {reader_id} has read {sorted(seen)}; profile has {len(profile_tokens)} tokens")
print("top-5 unseen books by TF-IDF cosine:")
for c in top5_tfidf:
    print(f"  item {c.item_id:>5}  cosine {c.score:.3f}  genres {catalog[c.item_id].fields['genres']}")
print(f"library TF-IDF matrix and cosine scores match the by-hand ones: {library_matches}")

assert tf_checks_pass
assert np.allclose(np.linalg.norm(tfidf_by_hand, axis=1), 1.0)
assert library_matches

## Exercise 3 - Fit and register LexicalRetrievalPath, then read the scoreboard vs the floor

`LexicalRetrievalPath(keywords)` takes its corpus at construction, so `fit(rows, catalog=catalog_ids)` has the same signature as every other `RetrievalPath`; `fit` ignores the interactions (a content path indexes text, not the log) and restricts the index to catalog ids. We register the random floor, popularity, and lexical paths in one `PathRegistry` and score each on `val` with the Unit 1 scoreboard, `k=10`, cold readers excluded.

**Honest reading.** Lexical retrieval scores hit@10 of about 0.158 against a seeded random floor near 0.012 -- roughly 13x the floor -- and about 1.46x the popularity path. Clearing a 10x bar establishes that keyword overlap with what a reader has already read carries real signal about what they will read next; it does **not** establish that this is a strong recommender in absolute terms (hit@10 of 0.158 means roughly five readers in six get no hit in their top 10), nor that it will survive against the collaborative paths of later units, which the lesson expects to sit above it. The keyword text correlates with genres, so this is a content candidate source in the same family as simple genre matching. This number says nothing about BM25 versus TF-IDF cosine: the lesson measured that comparison separately and found the two to be peers on this corpus.

In [ ]:
lexical = bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=0)
registry = bookrec.PathRegistry()
for path in (random_floor, popularity, lexical):
    registry.register(path)


def score(path):
    return bookrec.run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
    )


lexical_board = score(lexical)
pop_board = score(popularity)
random_board = score(random_floor)
readers_scored = lexical_board.readers
lexical_hit = lexical_board.hit_rate_at_k
vs_floor = lexical_hit / random_board.hit_rate_at_k
vs_popularity = lexical_hit / pop_board.hit_rate_at_k
beats_floor_explanation = (
    "Lexical hit@10 is about 13x the seeded random floor and about 1.5x popularity, so keyword overlap "
    "with a reader's history carries real signal and the path earns its place as a content candidate "
    "source, in the same family as simple genre matching. Clearing 10x does not make it a strong "
    "recommender in absolute terms (most readers still get no hit in their top 10) and says nothing "
    "about how it will fare against the collaborative paths of later units, which should sit above it. "
    "It also says nothing about BM25 versus TF-IDF cosine; the lesson found those two to be peers here."
)

print(f"readers scored: {readers_scored}")
print(f"{'path':>12} {'hit@10':>8} {'recall@10':>10} {'vs floor':>9}")
for label, board in (("random", random_board), ("popularity", pop_board), ("lexical", lexical_board)):
    print(f"{label:>12} {board.hit_rate_at_k:>8.4f} {board.recall_at_k:>10.4f} "
          f"{board.hit_rate_at_k / random_board.hit_rate_at_k:>8.1f}x")
print(f"lexical / random = {vs_floor:.2f}x    lexical / popularity = {vs_popularity:.2f}x")
print()
print(beats_floor_explanation)

assert readers_scored > 0 and random_board.readers == pop_board.readers == readers_scored
assert vs_floor > 10, "lexical retrieval should beat the random floor by more than 10x on val"
assert vs_popularity > 1, "on this seed the lexical path should also sit above popularity"

## Exercise 4 - The b length-normalization effect on a long and a short document

Inside BM25's saturating term the denominator is $\mathrm{tf} + k_1\,(1 - b + b\,|d|/\mathrm{avgdl})$. At $b = 0$ the length factor is 1 for every document, so a 40-token bag and a 12-token bag are judged purely on their counts; at $b = 1$ the factor is $|d|/\mathrm{avgdl}$, so a document longer than average needs proportionally larger counts to earn the same score.

We locate the longest and shortest documents (ties broken by smaller id), build indexes at $b = 0$ and $b = 1$, and score reader 7's profile with each. `rank_of` is a stable descending argsort, so rank 1 is the best score and ties keep row order. Both chosen documents share only a few tokens with reader 7's profile, so they sit far down the ranking at either $b$ -- what matters is the *direction* they move. Finally we repeat the lesson's whole-top-10 measurement: the mean length of reader 7's ten recommended books at $b = 0$ versus $b = 1$.

As $b$ goes from 0 to 1 the long document's rank gets *worse* (a larger number) and the short document's gets *better*: that is exactly the correction $b$ exists to make, because a long bag matches more query tokens just by having more tokens, and length normalization charges it for that advantage.

In [ ]:
lengths_by_id = {item_id: len(tokenize(keywords[item_id])) for item_id in catalog_ids}
long_id = min(catalog_ids, key=lambda item_id: (-lengths_by_id[item_id], item_id))
short_id = min(catalog_ids, key=lambda item_id: (lengths_by_id[item_id], item_id))
long_len, short_len = lengths_by_id[long_id], lengths_by_id[short_id]
avgdl = float(np.mean(list(lengths_by_id.values())))

index_b0 = bookrec.BM25Index(keywords, b=0.0)
index_b1 = bookrec.BM25Index(keywords, b=1.0)
scores_b0 = index_b0.score(profile_tokens)
scores_b1 = index_b1.score(profile_tokens)
row_of = {item_id: row for row, item_id in enumerate(index.item_ids)}   # same row order in every index


def rank_of(item_id, scores):
    order = np.argsort(-scores, kind="stable")          # best score first; ties keep row order
    position = {int(row): rank for rank, row in enumerate(order, start=1)}
    return position[row_of[item_id]]


long_score_b0, long_score_b1 = float(scores_b0[row_of[long_id]]), float(scores_b1[row_of[long_id]])
short_score_b0, short_score_b1 = float(scores_b0[row_of[short_id]]), float(scores_b1[row_of[short_id]])
long_rank_b0, long_rank_b1 = rank_of(long_id, scores_b0), rank_of(long_id, scores_b1)
short_rank_b0, short_rank_b1 = rank_of(short_id, scores_b0), rank_of(short_id, scores_b1)


def mean_top10_length(scores):
    pool = [bookrec.Candidate(item_id, float(scores[row]), "bm25") for row, item_id in enumerate(index.item_ids)]
    top10 = bookrec.rank(pool, n=10, exclude=seen)
    return float(np.mean([lengths_by_id[c.item_id] for c in top10]))


mean_top10_len_b0 = mean_top10_length(scores_b0)
mean_top10_len_b1 = mean_top10_length(scores_b1)
b_effect_explanation = (
    "Going from b = 0 to b = 1 the long document's rank gets worse and the short document's gets better, "
    "because b charges a long bag for the extra matches it collects simply by having more tokens -- "
    "removing the length advantage is precisely the correction b exists to make."
)

print(f"longest document: item {long_id} ({long_len} tokens)   shortest: item {short_id} ({short_len} tokens)   "
      f"avgdl = {avgdl:.2f} (index.avgdl = {index.avgdl:.2f})")
print(f"{'document':>10} {'score b=0':>10} {'score b=1':>10} {'rank b=0':>9} {'rank b=1':>9}")
print(f"{'long':>10} {long_score_b0:>10.3f} {long_score_b1:>10.3f} {long_rank_b0:>9} {long_rank_b1:>9}")
print(f"{'short':>10} {short_score_b0:>10.3f} {short_score_b1:>10.3f} {short_rank_b0:>9} {short_rank_b1:>9}")
print(f"reader {reader_id} top-10 mean length: b=0 -> {mean_top10_len_b0:.1f} tokens, "
      f"b=1 -> {mean_top10_len_b1:.1f} tokens (avgdl {avgdl:.1f})")
print()
print(b_effect_explanation)

assert np.isclose(avgdl, index.avgdl) and long_len > avgdl > short_len
assert long_rank_b1 > long_rank_b0 and short_rank_b1 < short_rank_b0, "b must penalize the long document"
assert long_score_b1 < long_score_b0 and short_score_b1 > short_score_b0
assert mean_top10_len_b1 < mean_top10_len_b0, "length normalization should shorten the recommended books"

## Exercise 5 - Compute one BM25 score by hand and match BM25Index.score

For one document the formula is a sum over the distinct query tokens, each weighted by how often it appears in the query:

$$\mathrm{BM25}(q, d) = \sum_{t \in q} q_t \cdot \mathrm{idf}_t \cdot \frac{\mathrm{tf}_{d,t}\,(k_1 + 1)}{\mathrm{tf}_{d,t} + k_1\left(1 - b + b\,\dfrac{|d|}{\mathrm{avgdl}}\right)}.$$

The length term in the denominator depends only on the document, so we compute it once. The library skips query tokens outside the vocabulary, so `query_tf` is a `Counter` restricted to `index.vocab` (on this corpus every profile token is a catalog token, so nothing is dropped). We loop over the distinct tokens, read `tf` from the stored matrix, and accumulate `q_count * idf * saturated_tf`; the result matches `index.score(profile_tokens)[row]` to floating-point tolerance.

The longest document shares only two distinct tokens with reader 7's profile, and one of the two has tf = 1. To see the pattern the lesson was pointing at -- that saturation barely matters because matched terms are almost all tf = 1 -- we also tabulate the contributing (query token, document) pairs over reader 7's top-10 recommended books.

In [ ]:
index = bookrec.BM25Index(keywords)
row = row_of[long_id]
query_tf = Counter(token for token in profile_tokens if token in index.vocab)

length_term = index.k1 * (1.0 - index.b + index.b * index.doc_len[row] / index.avgdl)
bm25_by_hand = 0.0
contributing_terms = 0
contributing_tf1 = 0
for token, q_count in query_tf.items():
    col = index.vocab[token]
    tf_dt = index.tf[row, col]
    bm25_by_hand += q_count * index.idf[col] * tf_dt * (index.k1 + 1.0) / (tf_dt + length_term)
    if tf_dt > 0:
        contributing_terms += 1
        if tf_dt == 1:
            contributing_tf1 += 1
bm25_lib = float(index.score(profile_tokens)[row])
hand_matches_library = bool(np.isclose(bm25_by_hand, bm25_lib))

# the same tally over reader 7's top-10 recommended books, to see how rare tf > 1 is among matches
default_scores = index.score(profile_tokens)
top10_default = bookrec.rank(
    [bookrec.Candidate(item_id, float(default_scores[r]), "bm25") for r, item_id in enumerate(index.item_ids)],
    n=10, exclude=seen,
)
matched_pairs = [index.tf[row_of[c.item_id], index.vocab[token]] for c in top10_default for token in query_tf]
matched_pairs = [tf_value for tf_value in matched_pairs if tf_value > 0]
share_tf1_top10 = sum(1 for tf_value in matched_pairs if tf_value == 1) / len(matched_pairs)

print(f"document {long_id}: |d| = {int(index.doc_len[row])}, avgdl = {index.avgdl:.2f}, "
      f"k1 = {index.k1}, b = {index.b}, length term = {length_term:.4f}")
print(f"distinct query tokens: {len(query_tf)} (of {len(profile_tokens)} profile tokens, none outside the vocabulary)")
print(f"BM25 by hand = {bm25_by_hand:.6f}   library = {bm25_lib:.6f}   agree: {hand_matches_library}")
print(f"query tokens present in this document: {contributing_terms}, of which tf = 1: {contributing_tf1}")
print(f"over reader {reader_id}'s top-10 books: {len(matched_pairs)} matched (token, book) pairs, "
      f"{share_tf1_top10:.0%} of them with tf = 1")

assert hand_matches_library
assert contributing_terms > 0 and contributing_tf1 <= contributing_terms
assert share_tf1_top10 > 0.5, "most matched terms should have tf = 1, which is why k1 saturation is a small effect"

## Exercise 6 - Reader-dependent retrieval: shipped path vs by-hand ranking, exclusions, and no history

`LexicalRetrievalPath.retrieve` ignores the `query` id and reads the reader from `context["seen"]`: it concatenates the seen books' keyword bags, scores every document with BM25, drops the seen books, and returns the top `k`. Three checks confirm that description against the path fitted in Exercise 3. First, the shipped top-5 equals the top-5 we get by wrapping `index.score(profile_tokens)` in `Candidate`s and calling `bookrec.rank(..., exclude=seen)` -- same ids, same order. Second, none of reader 7's seen books leak into the result, and reader 11 (a different history) gets a different list: this is the first path in the registry whose output depends on who is asking. Third, an empty `seen` set yields `[]`.

Returning nothing for a reader with no history is the honest answer because the path has no profile to score against -- any list it produced would be a guess dressed up as a match -- and this is exactly where Unit 2's `PopularityRetrievalPath` steps in, since popularity needs no history at all.

In [ ]:
shipped_top5 = lexical.retrieve(query=reader_id, context={"seen": seen}, k=5)
shipped_ids = [c.item_id for c in shipped_top5]
by_hand_top5 = bookrec.rank(
    [bookrec.Candidate(i, float(s), "bm25") for i, s in zip(index.item_ids, index.score(profile_tokens))],
    n=5, exclude=seen,
)
by_hand_ids = [c.item_id for c in by_hand_top5]
same_ranking = shipped_ids == by_hand_ids

no_seen_leaked = not (seen & set(shipped_ids))
other_reader = 11
other_ids = [c.item_id for c in lexical.retrieve(other_reader, {"seen": seen_by[other_reader]}, 5)]
reader_dependent = other_ids != shipped_ids

no_history = lexical.retrieve(query=999_999, context={"seen": set()}, k=5)
no_history_is_empty = no_history == []
cold_start_explanation = (
    "With no seen books there is no keyword profile to score, so an empty list is the honest answer "
    "(anything else would be a guess dressed up as a match); Unit 2's PopularityRetrievalPath, which "
    "needs no history, is the path that serves such a reader."
)

print(f"reader {reader_id} shipped top-5: {shipped_ids}")
print(f"reader {reader_id} by-hand top-5: {by_hand_ids}   same ranking: {same_ranking}")
print(f"seen books leaked: {not no_seen_leaked}   reader {other_reader} top-5: {other_ids}   "
      f"reader-dependent: {reader_dependent}")
print(f"no history -> {no_history!r}   empty: {no_history_is_empty}")
print(f"overlap between TF-IDF cosine (Exercise 2) and BM25 top-5 for reader {reader_id}: "
      f"{sorted(set(top5_tfidf_ids) & set(shipped_ids))}")
print()
print(cold_start_explanation)

assert same_ranking, "the shipped path must reproduce the by-hand BM25 ranking"
assert no_seen_leaked and reader_dependent and no_history_is_empty
assert len(shipped_ids) == 5 and len(other_ids) == 5

## Exercise 7 - Challenge: the two k1 limits -- linear tf and binary presence

Write $L = 1 - b + b\,|d|/\mathrm{avgdl}$ and $s(k_1) = \dfrac{\mathrm{tf}\,(k_1 + 1)}{\mathrm{tf} + k_1 L}$.

**$k_1 \to \infty$.** Divide numerator and denominator by $k_1$: $s = \dfrac{\mathrm{tf}\,(1 + 1/k_1)}{\mathrm{tf}/k_1 + L} \to \dfrac{\mathrm{tf}}{L}$. At $b = 0$, $L = 1$ and BM25 becomes $\sum_t q_t\,\mathrm{idf}_t\,\mathrm{tf}_{d,t}$: raw TF-IDF summed over the query, with no cosine normalization.

**$k_1 \to 0$.** Set $k_1 = 0$ directly: $s = \dfrac{\mathrm{tf}}{\mathrm{tf}} = 1$ whenever $\mathrm{tf} > 0$ (and the library defines the $0/0$ case as 0), so BM25 becomes $\sum_t q_t\,\mathrm{idf}_t\,[\mathrm{tf}_{d,t} > 0]$: binary term presence, with the length factor multiplied away by $k_1 = 0$.

Numerically, `BM25Index(keywords, k1=1e6, b=0.0)` matches the linear sum to a relative tolerance well inside $10^{-3}$ (the residual is of order $1/k_1$), and `BM25Index(keywords, k1=0.0)` matches the presence sum exactly.

On the scoreboard the $k_1 = 0$ limit lands at hit@10 of about 0.170 against the default's 0.158 -- at the top of the narrow band the lesson reported for every BM25 setting, not a different regime. Binary presence is about as good as it gets here because the keyword bags are ~26 tokens long and nearly every matched term has tf = 1 (Exercise 5), so the count carries almost no information beyond presence and the saturation curve has nothing to shape; $k_1$ would matter on a corpus of long documents -- full reviews or chapters -- where a term repeated ten times is genuinely stronger evidence than a term that appears once and the question of *how much* stronger has to be answered.

In [ ]:
linear_limit_derivation = (
    "Divide top and bottom of tf (k1 + 1) / (tf + k1 L) by k1: tf (1 + 1/k1) / (tf/k1 + L) -> tf / L as "
    "k1 -> infinity; at b = 0, L = 1 and BM25 is the linear sum of q_t * idf_t * tf_dt."
)
presence_limit_derivation = (
    "Put k1 = 0: tf (0 + 1) / (tf + 0) = tf / tf = 1 for every tf > 0 (and 0 when tf = 0), so BM25 is "
    "the sum of q_t * idf_t over query tokens present in the document, with length ignored."
)

index_big_k1 = bookrec.BM25Index(keywords, k1=1e6, b=0.0)
linear_scores = np.zeros(len(index.item_ids))
presence_scores = np.zeros(len(index.item_ids))
for token, q_count in query_tf.items():
    col = index.vocab[token]
    linear_scores += q_count * index.idf[col] * index.tf[:, col]
    presence_scores += q_count * index.idf[col] * (index.tf[:, col] > 0)
big_k1_scores = index_big_k1.score(profile_tokens)
big_k1_is_linear = bool(np.allclose(big_k1_scores, linear_scores, rtol=1e-3))

index_k1_zero = bookrec.BM25Index(keywords, k1=0.0)
k1_zero_scores = index_k1_zero.score(profile_tokens)
k1_zero_is_presence = bool(np.allclose(k1_zero_scores, presence_scores))

k1_zero_board = score(bookrec.LexicalRetrievalPath(keywords, k1=0.0, name="bm25-k1-0").fit(rows, catalog=catalog_ids))
k1_zero_hit = k1_zero_board.hit_rate_at_k
default_hit = lexical_board.hit_rate_at_k
why_presence_suffices = (
    "On ~26-token keyword bags nearly every matched term has tf = 1, so the count carries almost no "
    "information beyond presence and the saturation curve has nothing to shape -- binary presence "
    "weighted by idf already captures the match. k1 would matter on a corpus of long documents such as "
    "full reviews or chapters, where a term repeated ten times is genuinely stronger evidence than one "
    "appearance and the question of how much stronger needs an answer."
)

matched = linear_scores > 0
# the saturating term never exceeds tf / L, so the finite-k1 score sits at or below the linear sum
rel_err = np.max((linear_scores[matched] - big_k1_scores[matched]) / linear_scores[matched])
print(linear_limit_derivation)
print(presence_limit_derivation)
print()
print(f"k1 = 1e6, b = 0 vs linear tf-idf sum: allclose(rtol=1e-3) = {big_k1_is_linear}   "
      f"(max relative error {rel_err:.1e})")
print(f"k1 = 0 vs binary presence sum: allclose = {k1_zero_is_presence}")
print(f"val hit@10: bm25-k1-0 = {k1_zero_hit:.4f}   default bm25 (k1=1.2, b=0.75) = {default_hit:.4f}")
print()
print(why_presence_suffices)

assert big_k1_is_linear and k1_zero_is_presence
assert k1_zero_hit >= default_hit, "the binary-presence limit should sit at the top of, or level with, the band"
assert k1_zero_hit / random_board.hit_rate_at_k > 10

## Exercise 8 - Challenge: why a rare keyword dominates a match

From the formula, a token in a single book has $\mathrm{df} = 1$ and earns $\log\!\big(1 + (N - 0.5)/1.5\big) \approx \log N$ -- about 7.2 for $N = 2000$, against $\log N \approx 7.6$ -- while a token in nearly every book has $N - \mathrm{df} \approx 0$ and earns $\log(1 + \text{small}) \approx 0$, so in general the ratio of the rarest to the commonest idf is unbounded. **On this corpus the ratio is modest**: the keyword generator keeps every token in between 42 and 113 of the 2000 books, so idf runs only from about 2.87 to 3.85, and among reader 7's profile tokens the rarest (`df` 48) and the commonest (`df` 106) differ by a factor of about 1.27. We report the measured ratio honestly and keep the formula's extremes as the explanation of what idf *would* do on a corpus with genuinely rare and genuinely ubiquitous words.

Even a 1.3x spread shows up in a match. Decomposing the top-1 book's BM25 score into per-token contributions (the same stored-array computation as Exercise 5, one term per contributing query token) reproduces `index.score` for that book; the largest single contribution comes from `limestone`, a mid-`df` token that reader 7's profile repeats three times (`q_count` = 3), and that one token outweighs every other contributing token -- query repetition and idf together, not raw document `tf`, decide the match.

The two-token query `[rare_token, common_token]` ranks first a book that contains the rare token (several times). A recommender scoring by raw `tf` alone would let whichever word a book repeats most decide the match regardless of how widespread that word is, so the commonest keyword in the catalog would drive recommendations and every reader would be pushed toward the same books that repeat it.

In [ ]:
profile_distinct = sorted(set(profile_tokens))
rare_token = max(profile_distinct, key=lambda token: index.idf[index.vocab[token]])
common_token = min(profile_distinct, key=lambda token: index.idf[index.vocab[token]])
rare_df, rare_idf = int(index.df[index.vocab[rare_token]]), float(index.idf[index.vocab[rare_token]])
common_df, common_idf = int(index.df[index.vocab[common_token]]), float(index.idf[index.vocab[common_token]])
idf_ratio = rare_idf / common_idf
n_books = len(index.item_ids)
idf_ratio_explanation = (
    f"With N = {n_books}, a token in one book would earn log(1 + (N - 0.5) / 1.5) = "
    f"{np.log(1.0 + (n_books - 0.5) / 1.5):.2f} (about log N) while a token in nearly every book earns "
    f"close to 0, so the formula makes the ratio unbounded in general; here df only spans "
    f"{int(index.df.min())}..{int(index.df.max())}, so the measured ratio is a modest {idf_ratio:.2f}."
)

top1_id = shipped_ids[0]
top1_row = row_of[top1_id]
length_term_top1 = index.k1 * (1.0 - index.b + index.b * index.doc_len[top1_row] / index.avgdl)
contributions = {}
for token, q_count in query_tf.items():
    col = index.vocab[token]
    tf_dt = index.tf[top1_row, col]
    if tf_dt > 0:
        saturated_tf = tf_dt * (index.k1 + 1.0) / (tf_dt + length_term_top1)
        contributions[token] = float(q_count * index.idf[col] * saturated_tf)
top1_total = float(index.score(profile_tokens)[top1_row])
contributions_sum_matches = bool(np.isclose(sum(contributions.values()), top1_total))
dominant_token = max(contributions, key=lambda token: contributions[token])
dominant_df = int(index.df[index.vocab[dominant_token]])
dominant_share = contributions[dominant_token] / top1_total
tokens_outweighed = sum(1 for value in contributions.values() if value < contributions[dominant_token])

two_token_scores = index.score([rare_token, common_token])
two_token_top_id = index.item_ids[int(np.argsort(-two_token_scores, kind="stable")[0])]
top_contains_rare = rare_token in tokenize(keywords[two_token_top_id])
raw_tf_failure = (
    "Scoring by raw tf alone lets whichever word a book repeats most decide the match regardless of how "
    "widespread that word is, so the commonest keyword in the catalog would drive recommendations and "
    "push every reader toward the same books that repeat it."
)

print(f"rarest profile token: {rare_token!r} df={rare_df} idf={rare_idf:.3f}   "
      f"commonest: {common_token!r} df={common_df} idf={common_idf:.3f}   ratio {idf_ratio:.3f}")
print(idf_ratio_explanation)
print()
print(f"top-1 unseen book for reader {reader_id}: item {top1_id} ({int(index.doc_len[top1_row])} tokens); "
      f"{len(contributions)} contributing tokens sum to {sum(contributions.values()):.4f} "
      f"vs index.score {top1_total:.4f}: {contributions_sum_matches}")
for token, value in sorted(contributions.items(), key=lambda pair: -pair[1])[:5]:
    print(f"  {token:>10} df={int(index.df[index.vocab[token]]):>4} q_count={query_tf[token]} "
          f"tf={int(index.tf[top1_row, index.vocab[token]])} contribution {value:.3f}")
print(f"dominant token {dominant_token!r}: df={dominant_df}, share {dominant_share:.1%}, "
      f"outweighs {tokens_outweighed} of the other {len(contributions) - 1} contributing tokens")
print()
print(f"query [{rare_token!r}, {common_token!r}] -> top book item {two_token_top_id}, "
      f"contains the rare token: {top_contains_rare} "
      f"(tf of rare token there: {int(index.tf[row_of[two_token_top_id], index.vocab[rare_token]])})")
print(raw_tf_failure)

assert rare_idf > common_idf and idf_ratio > 1
assert contributions_sum_matches, "per-token contributions must add up to the library's BM25 score"
assert tokens_outweighed == len(contributions) - 1 and dominant_share > 1 / len(contributions)
assert top_contains_rare